# Analyst-weighted revisions: final project presentation

This notebook supports direct revision signals and the earlier z-score strategy. The run configuration and report banner identify the actual saved mode.

The new **revision_signal** configuration uses only positive-Sharpe analysts, normalizes their weights to sum to one per stock, goes long when the raw weighted revision is **above +1**, and short when it is **below −1**. There is no z-score transformation or threshold-crossing requirement. The shared limit is **200 positions**, held for **14 trading sessions**.

Synthetic results validate software only; they are not empirical performance.


## 1. Run instructions

1. Select a Python kernel containing the project and notebook dependencies. From the workspace terminal: `.venv/bin/python -m pip install -r analyst_recommendation_anomaly/requirements-notebook.txt`.
2. Edit only the settings cell below, then choose **Restart Kernel and Run All**.
3. Use `RUN_MODE="demo"` to test everything, `"real"` to run licensed inputs, or `"existing"` to analyze a completed run directory.
4. For real data, obtain daily inputs with `zsh analyst_recommendation_anomaly/download_analyst_daily.command` from the workspace directory. Set explicit extract paths below; a unique completed extract can be discovered automatically.
5. Tables and PNG/PDF figures are exported automatically. The terminal runner also exports a standalone HTML presentation and executed notebook: `python3 analyst_recommendation_anomaly/scripts/run_presentation.py --mode demo`.

Existing monthly CRSP files are **not** sufficient. Real mode fails if required inputs are missing; it never falls back to synthetic data. Actual start/end dates are shown in the results, and pre-start observations are retained for signal warmup.

In [ ]:
# Change these settings, then Run All. Paths may be absolute or project-relative.
RUN_MODE = "real"                 # demo | real | existing
EXTRACT_DIR = None                 # e.g. data/raw/ibes_analyst_...
MARKET_DIR = None                  # e.g. data/raw/analyst_daily_...
EXISTING_RUN_DIR = None            # required only for existing mode
START = "2005-01-03"               # real-mode evaluation dates (example, not optimized)
END = "2024-12-31"
DOWNLOAD_DAILY = False             # explicit opt-in; needs WRDS credentials/entitlements
DOWNLOAD_START = "2000-01-01"      # warmup history
RUN_SENSITIVITY = False             # one-factor cost/threshold scenarios; never changes baseline
REPORT_DIR = None                  # default: the run's presentation/ subdirectory
STRATEGY_PYTHON = None             # default: workspace .venv if available, else kernel Python
CONFIG_OVERRIDES = {
    "strategy_mode": "revision_signal",
    "entry_signal_threshold": 1.0,
    "revision_freshness_sessions": None,
    "missing_terminal_policy": "last_close",
    "top_n": 200,
    "min_active_analysts": 2,
    "negative_scores": "exclude",
    "portfolio_mode": "long_short",
    "short_borrow_bps_annual": 0.0,
}


In [ ]:
from pathlib import Path
from dataclasses import asdict
import importlib.metadata as metadata
import json
import subprocess
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display, HTML, Markdown

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT = next((p for base in candidates for p in (base, base / "analyst_recommendation_anomaly")
                if (p / "src" / "revision_zscore.py").exists()), None)
if PROJECT is None:
    raise RuntimeError("Open this notebook from the project workspace.")
sys.path.insert(0, str(PROJECT))
from src.analyst_sharpe import AnalystSharpeConfig, prepare_market
from src.presentation import (LABELS, load_run, return_panel, performance_table,
    drawdown_panel, calendar_returns, rolling_statistics, trading_table,
    closed_trade_returns, audit_table, order_table, replay_sensitivity)

settings = json.loads((PROJECT / "configs/analyst_sharpe.json").read_text())
settings.update(CONFIG_OVERRIDES)
config = AnalystSharpeConfig(**settings)
if config.strategy_mode not in ("revision_zscore", "revision_signal"):
    raise ValueError("This presentation requires a revision strategy.")
def resolve(value):
    if value is None: return None
    p = Path(value).expanduser()
    return p if p.is_absolute() else PROJECT / p

venv_python = PROJECT.parent / ".venv" / "bin" / "python"
strategy_python = str(resolve(STRATEGY_PYTHON) if STRATEGY_PYTHON else
                      venv_python if venv_python.exists() else sys.executable)
COLORS = {"analyst_sharpe": "#126782", "unweighted": "#E07A35", "benchmark": "#755AA5"}
plt.rcParams.update({"figure.figsize": (12, 5), "axes.spines.top": False,
                    "axes.spines.right": False, "axes.grid": True, "grid.alpha": .18,
                    "font.size": 11, "figure.dpi": 110, "savefig.dpi": 160})
pd.options.display.max_columns = 30
pd.options.display.float_format = lambda x: f"{x:,.4f}"
display(HTML("<style>.dataframe th{text-align:left!important}.dataframe td{padding:6px 12px!important}</style>"))
print("Project:", PROJECT)
print("Notebook Python:", sys.executable)
print("Strategy Python:", strategy_python)

## 2. Method and assumptions

Ratings 1–5 map to +2, +1, 0, −1, −2. Each analyst's revision is the latest available rating minus its previous known rating for the same stock. The latest revision persists without an age cutoff until replaced or invalidated.

For **revision_signal**, retain only analysts with strictly positive historical skill and a valid revision. Require at least two such analysts per stock. Normalize weights within that stock:

$$w_{i,a,t}=\frac{S_{a,t}}{\sum_{b:S_{b,t}>0} S_{b,t}},\qquad Q_{i,t}=\sum_a w_{i,a,t}\Delta R_{i,a,t},\qquad \sum_a w_{i,a,t}=1.$$

Long candidates have $Q>1$; short candidates have $Q<-1$. Equality does not qualify. The unweighted comparison averages revisions over the same positive-Sharpe contributor pool. Candidates qualify on each day beyond the threshold; no rolling-window warmup or crossing is required. Rank competing entries by absolute signal magnitude, then PERMNO.

For saved **revision_zscore** runs, the original prior-20-session standardization and crossings above +1.68 / below −1.68 apply; the original scored-analyst eligibility convention is retained. Do not interpret historical z-score results as direct-signal results.

Skill remains mean/sample standard deviation of rating-direction-adjusted 30-session stock-minus-market returns, with 25 completed directional calls and expanding history. This is the existing historical skill definition, not a revision-specific score.

Trade at the next close and hold for 14 subsequent trading sessions. The raw-signal configuration allows 200 positions across both sides; new positions target 0.5% of NAV, constrained by the shared gross-exposure budget including costs. Positions are never resized or reset while held. A persistent qualifying signal may re-enter on the session after exit; there is no exit-and-reentry on the same close. There is no fixed 50/50 split or dollar neutrality.

Universe, availability dates, 10 bps execution costs, zero-interest cash, initial NAV of $10 million, and no final forced liquidation are unchanged. Shorts are signed total-return liabilities; configured borrow fees default to zero, and locates, recalls, rebates, and margin liquidations are not modeled.

Approved terminal and off-exchange last-value exit approximations remain explicitly flagged. The verified BIIB (2020-11-06) and SOV (2005-10-24) halts carry valuation without execution and retains its reopening return. Other unresolved missing held returns fail. These exceptions do not fill historical analyst labels or signals. Full assumptions are in [the strategy specification](../analyst_sharpe_strategy.md).


In [ ]:
def discover_unique(pattern):
    found = []
    for path in sorted((PROJECT / "data" / "raw").glob(pattern)):
        manifest_path = path / "manifest.json"
        if manifest_path.exists() and json.loads(manifest_path.read_text()).get("complete"):
            found.append(path)
    if len(found) != 1:
        raise ValueError(f"Expected exactly one completed {pattern} extract; found {len(found)}. Set an explicit path.")
    return found[0]

if RUN_MODE not in {"demo", "real", "existing"}:
    raise ValueError("RUN_MODE must be demo, real or existing")
if RUN_MODE == "existing":
    if EXISTING_RUN_DIR is None:
        raise ValueError("Set EXISTING_RUN_DIR to a completed revision-zscore run.")
    RUN_DIR = resolve(EXISTING_RUN_DIR)
else:
    if RUN_MODE == "real":
        if DOWNLOAD_DAILY:
            from src.download_analyst_market import download
            MARKET_DIR = str(download(DOWNLOAD_START, END, PROJECT / "data" / "raw"))
        EXTRACT_DIR = resolve(EXTRACT_DIR) or discover_unique("ibes_analyst_*")
        MARKET_DIR = resolve(MARKET_DIR) or discover_unique("analyst_daily_*")
    run_configs = PROJECT / "outputs" / "analyst_sharpe" / "notebook_configs"
    run_configs.mkdir(parents=True, exist_ok=True)
    import tempfile
    with tempfile.NamedTemporaryFile(mode="w", suffix=".json", dir=run_configs, delete=False) as handle:
        json.dump(asdict(config), handle, indent=2)
        config_path = Path(handle.name)
    command = [strategy_python, "-u", "-m", "src.run_analyst_sharpe", "--config", str(config_path)]
    if RUN_MODE == "demo":
        command += ["--demo"]
    else:
        command += ["--extract", str(EXTRACT_DIR), "--market-dir", str(MARKET_DIR),
                    "--names", str(PROJECT / "data/raw/crsp_names.parquet"), "--start", START, "--end", END]
    print("Running:", " ".join(command), flush=True)
    from src.notebook_process import run_logged
    RUN_DIR = run_logged(command, PROJECT)

manifest, frames = load_run(RUN_DIR)
config = AnalystSharpeConfig(**manifest["config"])  # Always analyze the actual saved configuration.
RAW_SIGNAL = config.strategy_mode == "revision_signal"
ENTRY_THRESHOLD = config.entry_signal_threshold if RAW_SIGNAL else config.entry_zscore
SYNTHETIC = bool(manifest["synthetic"])
DATA_LABEL = "SYNTHETIC DEMO — NOT EMPIRICAL EVIDENCE" if SYNTHETIC else "LICENSED-DATA RESEARCH — EXPLORATORY"
VALUATION_EVENTS = frames.get("valuation_events", pd.DataFrame())
if len(VALUATION_EVENTS):
    DATA_LABEL += " — FLAGGED VALUATION EVENTS"
    print("Valuation events are disclosed below: verified halt carries retain reopening returns; off-exchange proxy exits assume liquidation at prior value despite no executable price.")
    display(VALUATION_EVENTS)
PROXY_EXITS = frames["completed_positions"].query("exit_reason in ['terminal_close_proxy', 'off_exchange_close_proxy']")
if len(PROXY_EXITS):
    DATA_LABEL += " — APPROXIMATE EXITS"
    print(f"CAUTION: {len(PROXY_EXITS)} strategy positions use approximate terminal or off-exchange exits. Performance depends on this approximation.")
    display(PROXY_EXITS)
REPORT = resolve(REPORT_DIR) or RUN_DIR / "presentation"
for folder in [REPORT, REPORT / "figures", REPORT / "tables"]:
    folder.mkdir(parents=True, exist_ok=True)
from src.run_analyst_sharpe import fingerprint
(REPORT / "analysis_context.json").write_text(json.dumps({"run_dir": str(RUN_DIR), "data_label": DATA_LABEL,
    "manifest": manifest, "analysis_code": [fingerprint(PROJECT / "src/presentation.py"),
        fingerprint(PROJECT / "notebooks/analyst_revision_presentation.ipynb")], "notebook_packages": {name: metadata.version(name) for name in
    ["numpy", "pandas", "matplotlib", "nbformat"]}}, indent=2))

def export_table(table, name):
    table.to_csv(REPORT / "tables" / f"{name}.csv", index=True)
    return table

def show_figure(fig, name):
    fig.suptitle(DATA_LABEL + "\n" + name.replace("_", " ").title(), fontsize=13, fontweight="bold")
    fig.tight_layout()
    fig.savefig(REPORT / "figures" / f"{name}.png", bbox_inches="tight")
    fig.savefig(REPORT / "figures" / f"{name}.pdf", bbox_inches="tight")
    plt.show()
    plt.close(fig)

export_table(PROXY_EXITS.query("exit_reason == 'terminal_close_proxy'"), "approximate_terminal_exits")
export_table(PROXY_EXITS, "approximate_exits")
if len(VALUATION_EVENTS):
    export_table(VALUATION_EVENTS, "valuation_events")
returns = return_panel(frames["daily_results"])
performance = export_table(performance_table(frames["daily_results"]), "performance")
trading = export_table(trading_table(frames), "trading")
closed = export_table(closed_trade_returns(frames), "closed_positions")
first, last = returns.index.min().date(), returns.index.max().date()
display(HTML(f"<div style='padding:16px;border:2px solid #126782;background:#eef6f8;color:#153641'><b>{DATA_LABEL}</b><br>"
             f"Observed sample: {first} to {last} · {len(returns):,} sessions<br>Threshold: {ENTRY_THRESHOLD} · "
             f"Holding period: {config.holding_sessions} sessions · Cost: {config.cost_bps:g} bps<br>"
             f"Signal mode: {config.strategy_mode} · Portfolio: {config.portfolio_mode} · Position limit: {config.top_n} · Annual short borrow: {config.short_borrow_bps_annual:g} bps</div>"))
print("Run:", RUN_DIR)
print("Report exports:", REPORT)

## 3. Executive scorecard

All series use exactly the same observed sessions. **Annualization uses 252 sessions**, including CAGR. Sharpe uses a zero risk-free rate; the benchmark is fully invested while the strategies can hold substantial cash. This is a comparison of realized research portfolios, not a factor-adjusted alpha claim.

In [ ]:
headline_columns = ["total_return", "cagr_252", "volatility_252", "sharpe_zero_rf", "max_drawdown", "information_ratio"]
headline = performance[headline_columns].rename(index=LABELS)
formats = {c: "{:.2%}" for c in ["total_return", "cagr_252", "volatility_252", "max_drawdown"]}
formats.update({"sharpe_zero_rf": "{:.2f}", "information_ratio": "{:.2f}"})
display(headline.style.format(formats, na_rep="N/A"))
display(trading[["entries", "completed_positions", "open_positions", "win_rate", "mean_cash_weight", "total_cost_dollars"]]
        .rename(index=LABELS).style.format({"win_rate":"{:.1%}", "mean_cash_weight":"{:.1%}", "total_cost_dollars":"${:,.0f}"}, na_rep="N/A"))

## 4. Cumulative performance and drawdowns

The upper panel shows the value of $1 invested before the first reported session, including trading costs from the first day. The lower panel shows the percentage decline from each series’ highest value reached so far, with the initial $1 included as a starting peak. A drawdown of −10% means wealth is 10% below that peak.

If the report is labeled **APPROXIMATE EXITS**, these curves include the disclosed last-close liquidation assumptions; they are not results based entirely on verified terminal payoffs.


In [ ]:
wealth = (1 + returns).cumprod()
drawdowns = drawdown_panel(returns)
export_table(wealth, "wealth")
export_table(drawdowns, "drawdowns")
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
for name in returns:
    initial_date = returns.index[0] - pd.Timedelta(1, unit="D")
    plotted = pd.concat([pd.Series([1.], index=[initial_date]), wealth[name]])
    axes[0].plot(plotted.index, plotted, color=COLORS[name], label=LABELS[name], linewidth=1.8)
    axes[1].plot(drawdowns.index, drawdowns[name], color=COLORS[name], label=LABELS[name])
axes[0].set_ylabel("Growth of USD 1")
axes[0].legend(loc="upper left")
axes[1].set_ylabel("Drawdown")
axes[1].yaxis.set_major_formatter(PercentFormatter(1))
show_figure(fig, "wealth_and_drawdown")

## 5. Complete risk and return statistics

Sortino uses the square root of the mean squared negative daily return as downside deviation, with target zero. Calmar is CAGR divided by absolute maximum drawdown. Expected shortfall is the mean daily return at or below the empirical fifth percentile. These are descriptive sample statistics, not forecasts or significance tests. Zero-denominator metrics are shown as N/A.

In [ ]:
display(performance.rename(index=LABELS).T.style.format("{:,.4f}", na_rep="N/A"))
rolling_sharpe, rolling_vol = rolling_statistics(returns, window=126)
export_table(rolling_sharpe, "rolling_126_session_sharpe")
export_table(rolling_vol, "rolling_126_session_volatility")
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
for name in returns:
    axes[0].plot(rolling_sharpe.index, rolling_sharpe[name], color=COLORS[name], label=LABELS[name])
    axes[1].plot(rolling_vol.index, rolling_vol[name], color=COLORS[name], label=LABELS[name])
axes[0].axhline(0, color="grey", linewidth=.8)
axes[0].set_ylabel("126-session Sharpe, zero rf")
axes[0].legend()
axes[1].set_ylabel("Annualized volatility")
axes[1].yaxis.set_major_formatter(PercentFormatter(1))
show_figure(fig, "rolling_risk")

## 6. Calendar-year and monthly performance

Compounded calendar returns retain partial first and last years/months. Session counts are included; partial periods must not be described as complete-year results. The heatmap leaves unobserved months blank.

In [ ]:
yearly = export_table(calendar_returns(returns, "Y"), "calendar_year_returns")
monthly = export_table(calendar_returns(returns, "M"), "calendar_month_returns")
display(yearly.rename(columns=LABELS).style.format({LABELS[c]:"{:.2%}" for c in returns}, na_rep="N/A"))
fig, ax = plt.subplots(figsize=(12, 5))
yearly[list(returns)].rename(columns=LABELS).plot.bar(ax=ax, color=[COLORS[c] for c in returns])
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.set_ylabel("Calendar-period return (edge periods may be partial)")
ax.tick_params(axis="x", rotation=0)
show_figure(fig, "calendar_year_returns")
heat = monthly[["analyst_sharpe"]].copy()
heat["year"], heat["month"] = heat.index.year, heat.index.month
heat = heat.pivot(index="year", columns="month", values="analyst_sharpe").reindex(columns=range(1,13))
fig, ax = plt.subplots(figsize=(12, max(3, len(heat)*.55 + 1.5)))
bound = max(.01, float(np.nanmax(np.abs(heat.to_numpy()))))
im = ax.imshow(heat.to_numpy(), cmap="RdYlGn", vmin=-bound, vmax=bound, aspect="auto")
ax.set_xticks(range(12), ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
ax.set_yticks(range(len(heat)), heat.index)
ax.grid(False)
for y in range(len(heat)):
    for x in range(12):
        value = heat.iloc[y, x]
        if pd.notna(value): ax.text(x, y, f"{value:.1%}", ha="center", va="center", fontsize=9)
fig.colorbar(im, ax=ax, format=PercentFormatter(1), label="Weighted strategy monthly return")
show_figure(fig, "monthly_return_heatmap")

## 7. Trading activity, cash allocation, and concentration

The strategy enters on events, not at fixed rebalance dates. In long/short mode cash includes short-sale proceeds and is not a measure of uninvested capital; gross and net exposures are shown separately. Position weights can drift above their entry targets because existing positions are not resized. Trade-level results below include fees and only completed positions; open trades are reported separately.

In [ ]:
display(trading.rename(index=LABELS).T.style.format("{:,.4f}", na_rep="N/A"))
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
for name, group in frames["daily_results"].groupby("strategy"):
    axes[0].plot(group.date, group.n_holdings, color=COLORS[name], label=LABELS[name])
    if config.portfolio_mode == "long_short":
        axes[1].plot(group.date, group.gross_exposure, color=COLORS[name], label=LABELS[name]+" gross")
        axes[1].plot(group.date, group.net_exposure, color=COLORS[name], linestyle="--", label=LABELS[name]+" net")
    else:
        axes[1].plot(group.date, 1-group.cash/group.nav, color=COLORS[name], label=LABELS[name])
axes[0].axhline(config.top_n, color="grey", linestyle="--", label="Position limit")
axes[0].set_ylabel("Open positions")
axes[0].legend()
axes[1].set_ylabel("Exposure / NAV")
axes[1].legend()
axes[1].yaxis.set_major_formatter(PercentFormatter(1))
show_figure(fig, "positions_and_investment")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, group in closed.groupby("strategy"):
    axes[0].hist(group.net_trade_return, bins=25, alpha=.5, color=COLORS[name], label=LABELS[name])
    axes[1].hist(group.held_sessions, bins=np.arange(0, config.holding_sessions+2)-.5,
                 alpha=.5, color=COLORS[name], label=LABELS[name])
axes[0].set_xlabel("Completed position return, including fees")
axes[0].xaxis.set_major_formatter(PercentFormatter(1))
axes[0].set_ylabel("Completed positions")
axes[1].set_xlabel("Sessions held after entry")
if not closed.empty: axes[0].legend()
show_figure(fig, "closed_trade_distributions")

if not closed.empty:
    attribution = closed.groupby(["strategy","permno"]).agg(
        completed_positions=("net_pnl","count"), closed_net_pnl=("net_pnl","sum"),
        mean_trade_return=("net_trade_return","mean"))
    export_table(attribution, "closed_position_pnl_by_stock")
    display(attribution.sort_values("closed_net_pnl", ascending=False).head(10))
else:
    display(Markdown("No completed positions in this sample; trade statistics are unavailable."))
display(Markdown("**Closed-position P&L attribution excludes unrealized P&L on open positions; it is not a full portfolio attribution.**"))
display(export_table(frames["open_positions"], "open_positions_at_end").head(20))

## 8. Costs and order decisions

Costs apply per dollar bought or sold, with no extra factor of one-half. The gross shadow portfolio follows the same accepted entries without costs. Its wealth gap versus net includes compounding and changes in affordable purchase amounts; it is not identical to the sum of fees.

In [ ]:
order_counts = export_table(order_table(frames["orders"]), "order_decisions")
display(order_counts)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for name, group in frames["daily_results"].groupby("strategy"):
    axes[0].plot(group.date, group.gross_nav/config.initial_nav, linestyle="--", color=COLORS[name], label=LABELS[name]+" gross")
    axes[0].plot(group.date, group.nav/config.initial_nav, color=COLORS[name], label=LABELS[name]+" net")
    axes[1].plot(group.date, group.cost_dollars.cumsum(), color=COLORS[name], label=LABELS[name])
axes[0].set_ylabel("Growth of USD 1")
axes[0].legend(fontsize=8)
axes[1].set_ylabel("Cumulative execution cost, USD")
axes[1].ticklabel_format(style="plain", axis="y")
show_figure(fig, "execution_costs")

## 9. Signal walkthrough and analyst diagnostics

This example selects the stock with the most accepted weighted entries in the observed sample, solely to illustrate the mechanics. It is not a stock recommendation or an unbiased performance case study. The prior-20-session mean excludes the current observation.

In [ ]:
accepted = frames["orders"].query("strategy == 'analyst_sharpe' and status == 'entry'")
example_stock = int(accepted.permno.value_counts().index[0]) if len(accepted) else int(frames["signals"].permno.iloc[0])
example = frames["signals"][frames["signals"].permno.eq(example_stock)].sort_values("date")
example = example[example.date.between(returns.index.min(), returns.index.max())]
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(example.date, example.signal, label="Weighted revision signal", color=COLORS["analyst_sharpe"])
if not RAW_SIGNAL:
    axes[0].plot(example.date, example.weighted_mean, label="Previous 20-session mean", color="grey", linestyle="--")
axes[0].set_ylabel("Revision units")
axes[0].legend()
axes[1].plot(example.date, example.signal if RAW_SIGNAL else example.weighted_z, color=COLORS["analyst_sharpe"])
axes[1].axhline(ENTRY_THRESHOLD, color=COLORS["unweighted"], linestyle="--", label=f"Long threshold {ENTRY_THRESHOLD}")
axes[1].axhline(-ENTRY_THRESHOLD, color=COLORS["unweighted"], linestyle=":", label=f"Short threshold {-ENTRY_THRESHOLD}")
example_orders = accepted[accepted.permno.eq(example_stock)]
axes[1].scatter(example_orders.cutoff, (example_orders.entry_signal if RAW_SIGNAL else example_orders.zscore), color="#277A46", marker="^", label="Accepted signal (trade next close)", zorder=4)
axes[1].set_ylabel("Raw revision signal" if RAW_SIGNAL else "Trailing z-score")
axes[1].legend(fontsize=9)
show_figure(fig, f"signal_walkthrough_permno_{example_stock}")

score_history = frames["analyst_scores"]
latest_scores = score_history[score_history.cutoff.eq(score_history.cutoff.max())]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(latest_scores.score, bins=30, color=COLORS["analyst_sharpe"], alpha=.8)
axes[0].set_xlabel("Analyst skill score at final cutoff")
axes[0].set_ylabel("Analysts")
if "revision_age_counts" in frames:
    age_counts = frames["revision_age_counts"]
    axes[1].bar(age_counts.age_sessions, age_counts.contributions, color=COLORS["unweighted"], alpha=.8)
else:
    ages = frames["contributions"].age_sessions
    axes[1].hist(ages, bins=30, color=COLORS["unweighted"], alpha=.8)
axes[1].set_xlabel("Announcement age in supplied calendar (lower bound for earlier calls)")
axes[1].set_ylabel("Analyst–stock–day contributions")
show_figure(fig, "analyst_skill_and_revision_age")

## 10. Coverage and implementation audit

Historical label exclusions are reported rather than hidden. Recent calls without matured outcomes are normal; missing stock outcomes or failed security links require investigation. An internal pass checks implementation consistency, not whether vendor timestamps or identifiers were historically correct. Checks on an empty set pass vacuously, so review actual trade and contribution counts too.

In [ ]:
audits = export_table(audit_table(frames, manifest["config"]), "implementation_audit")
display(audits)
if not audits.passed.all():
    raise AssertionError("Implementation audit failed; do not present the run as validated.")
coverage = export_table(frames["outcomes"].outcome_status.value_counts().rename_axis("status").to_frame("events"), "outcome_coverage")
display(coverage)
if manifest.get("input_diagnostics"):
    display(pd.Series(manifest["input_diagnostics"], name="input diagnostic").to_frame())
else:
    display(Markdown("Raw-link diagnostics are not applicable to the synthetic or pre-normalized input path."))
if "daily_signal_coverage" in frames:
    coverage_days = frames["daily_signal_coverage"].loc[returns.index.min():returns.index.max()]
else:
    signals_sample = frames["signals"][frames["signals"].date.between(returns.index.min(), returns.index.max())]
    coverage_days = signals_sample.groupby("date").agg(
        observed_stocks=("permno","nunique"), eligible_stocks=("eligible","sum"),
        valid_weighted_z=("weighted_z","count"), upward_crossings=("weighted_crossing","sum"))
export_table(coverage_days, "daily_signal_coverage")
fig, ax = plt.subplots(figsize=(12, 5))
for column in (["eligible_stocks", "upward_crossings"] if RAW_SIGNAL else ["eligible_stocks", "valid_weighted_z", "upward_crossings"]):
    ax.plot(coverage_days.index, coverage_days[column], label=("long threshold qualifications" if RAW_SIGNAL and column=="upward_crossings" else column.replace("_", " ")))
ax.set_ylabel("Stocks / signals per session")
ax.legend()
show_figure(fig, "signal_coverage")

## 11. Cost and threshold sensitivity — exploratory only

Sensitivity replays portfolio accounting while changing one threshold or cost assumption at a time. The threshold is in raw revision units for revision_signal and z-score units for revision_zscore. The baseline stays fixed at the saved configuration. These are in-sample diagnostics, not independent validation or a search for a winning setting. Disable RUN_SENSITIVITY to skip extra runs.


In [ ]:
sensitivity = pd.DataFrame()
if RUN_SENSITIVITY:
    if manifest.get("engine") == "streaming":
        from src.streaming_strategy import replay_saved_sensitivity
        sensitivity = replay_saved_sensitivity(RUN_DIR, config)
    else:
        if SYNTHETIC:
            from src.run_analyst_sharpe import demo_inputs
            _, daily_market, benchmark, _, _ = demo_inputs()
            # Refuse to replay a historical synthetic run against changed generator code.
            from src.run_analyst_sharpe import fingerprint
            saved_runner = next((item for item in manifest.get("code", []) if Path(item["path"]).name == "run_analyst_sharpe.py"), None)
            if saved_runner and saved_runner["sha256"] != fingerprint(PROJECT / "src/run_analyst_sharpe.py")["sha256"]:
                raise ValueError("Synthetic generator changed since this saved run; disable sensitivity or rerun the demo.")
        else:
            benchmark_input = next((Path(item["path"]) for item in manifest["inputs"] if Path(item["path"]).name == "benchmark.parquet"), None)
            if benchmark_input is None:
                raise ValueError("Original market data path is unavailable. Set RUN_SENSITIVITY=False.")
            market_root = benchmark_input.parent
            source = json.loads((market_root / "manifest.json").read_text())
            daily_market = pd.concat([pd.read_parquet(market_root / item["file"]) for item in source["files"]], ignore_index=True)
            benchmark = pd.read_parquet(benchmark_input)
        daily_market, benchmark = prepare_market(daily_market, benchmark)
        sensitivity = replay_sensitivity(frames, daily_market, benchmark, config, manifest["start"], manifest["end"])
    baseline = sensitivity[sensitivity.scenario.eq("baseline")].iloc[0]
    if not np.isclose(baseline.total_return, performance.loc["analyst_sharpe", "total_return"], rtol=1e-8, atol=1e-10):
        raise AssertionError("Sensitivity baseline does not reproduce the saved strategy. Check code/input versions.")
    export_table(sensitivity, "one_factor_sensitivity")
    display(sensitivity.style.format({"total_return":"{:.2%}", "cagr_252":"{:.2%}", "max_drawdown":"{:.2%}", "sharpe_zero_rf":"{:.2f}"}))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    threshold_cases = sensitivity[sensitivity.scenario.isin(["baseline", "threshold"])].sort_values("threshold")
    cost_cases = sensitivity[sensitivity.scenario.isin(["baseline", "cost"])].sort_values("cost_bps")
    axes[0].plot(threshold_cases.threshold, threshold_cases.total_return, marker="o", color=COLORS["analyst_sharpe"])
    axes[0].axvline(ENTRY_THRESHOLD, linestyle="--", color="grey")
    axes[0].set_xlabel("Entry threshold (units follow saved signal mode)")
    axes[1].plot(cost_cases.cost_bps, cost_cases.total_return, marker="o", color=COLORS["unweighted"])
    axes[1].set_xlabel("Cost per dollar traded, basis points")
    for ax in axes:
        ax.set_ylabel("Full-sample net return")
        ax.yaxis.set_major_formatter(PercentFormatter(1))
    show_figure(fig, "exploratory_sensitivity")
else:
    display(Markdown("Sensitivity was explicitly disabled; baseline analysis is unchanged."))

## 12. Presentation conclusions and limits

Use the generated observations below as a factual starting point. Separate statistical description from claims about investment skill. With synthetic data, the conclusion is about **software correctness only**.

For an empirical final project, the remaining research checks are historical identifier/timestamp validation, liquidity and execution realism, factor exposures, chronological holdout or walk-forward evaluation, dependence-aware uncertainty, and reporting of all tested settings. Overlapping analyst labels and overlapping portfolio holdings invalidate a naive independent-observation interpretation. Sector/size attribution is not shown because those point-in-time fields are not supplied to this daily strategy input.

The notebook discloses its approved missing-data valuation and exit approximations. It does not estimate p-values, sector exposures, or out-of-sample performance. Open positions remain marked to market at the sample end; closed-trade statistics exclude them.

In [ ]:
weighted = performance.loc["analyst_sharpe"]
comparison = performance.loc["unweighted"]
market_result = performance.loc["benchmark"]
trade_stats = trading.loc["analyst_sharpe"]
interpretation = "These values are synthetic and do not support an investment-performance claim." if SYNTHETIC else "These are exploratory empirical results; no out-of-sample skill claim has been established."
conclusions = f"""### Observed results — {DATA_LABEL}
- Observed sample: **{first} to {last}**, **{len(returns):,} sessions**.
- Weighted strategy total net return: **{weighted.total_return:.2%}**; unweighted: **{comparison.total_return:.2%}**; market benchmark: **{market_result.total_return:.2%}**.
- Weighted strategy annualized zero-rf Sharpe: **{weighted.sharpe_zero_rf:.2f}**; maximum drawdown: **{weighted.max_drawdown:.2%}**.
- **{int(trade_stats.entries):,} entries**, **{int(trade_stats.completed_positions):,} completed positions**, and **{int(trade_stats.open_positions):,} still open.
- Mean gross exposure: **{trade_stats.mean_gross_exposure:.1%}**; mean net exposure: **{trade_stats.mean_net_exposure:.1%}**; cumulative modeled costs: **USD {trade_stats.total_cost_dollars:,.0f}**.
- Internal implementation checks: **{int(audits.passed.sum())}/{len(audits)} passed**.

{interpretation}
"""
display(Markdown(conclusions))
(REPORT / "conclusions.md").write_text(conclusions)
export_table(pd.Series(manifest["config"], name="value").to_frame(), "run_configuration")
print("Presentation exports:", REPORT)
print("Figures: figures/*.png and figures/*.pdf")
print("Tables: tables/*.csv")
print("Run manifest:", RUN_DIR / "manifest.json")